In [2]:
import sys
sys.executable

'c:\\Users\\user\\Desktop\\SDIA5\\fake-news-DL\\FakeNews\\venv\\Scripts\\python.exe'

In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


^C


Looking in indexes: https://download.pytorch.org/whl/cu118
  Obtaining dependency information for torch from https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata (27 kB)
  Obtaining dependency information for torchvision from https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata (6.3 kB)
  Obtaining dependency information for torchaudio from https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp310-cp310-win_amd64.whl.metadata (6.8 kB)
  Obtaining dependency information for sympy>=1.13.3 from https://download.pytorch.org/whl/sympy-1.14.0-py3-none-any.whl.

   ------------------------------------- -- 2.6/2.8 GB 2.9 MB/s eta 0:00:59
   ------------------------------------- -- 2.6/2.8 GB 2.9 MB/s eta 0:00:59
   ------------------------------------- -- 2.6/2.8 GB 2.9 MB/s eta 0:01:00
   ------------------------------------- -- 2.6/2.8 GB 2.9 MB/s eta 0:01:00
   ------------------------------------- -- 2.6/2.8 GB 2.9 MB/s eta 0:01:01
   ------------------------------------- -- 2.6/2.8 GB 2.9 MB/s eta 0:01:01
   ------------------------------------- -- 2.6/2.8 GB 2.8 MB/s eta 0:01:01
   ------------------------------------- -- 2.6/2.8 GB 2.8 MB/s eta 0:01:02
   ------------------------------------- -- 2.6/2.8 GB 2.8 MB/s eta 0:01:02
   ------------------------------------- -- 2.6/2.8 GB 2.7 MB/s eta 0:01:03
   ------------------------------------- -- 2.6/2.8 GB 2.7 MB/s eta 0:01:04
   ------------------------------------- -- 2.6/2.8 GB 2.7 MB/s eta 0:01:04
   ------------------------------------- -- 2.6/2.8 GB 2.7 MB/s eta 0:01:05
   ---------

In [1]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0)


ModuleNotFoundError: No module named 'torch'

In [ ]:
!pip install transformers datasets sentencepiece

: 

In [ ]:
import pandas as pd
import re

df = pd.read_csv("/home/slim/FakeNews/data/WELFake_Dataset.csv")

df = df.rename(columns={
    "title": "title",
    "text": "text",
    "label": "label"
})

print(df.head())
print(df.label.value_counts(normalize=True))

: 

In [3]:
import re

def clean_text(t):
    if pd.isna(t):
        return ""
    t = str(t).lower()
    t = re.sub(r"http\S+", "", t)
    t = re.sub(r"[^a-zA-Z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text_clean"] = df["text"].apply(clean_text)
df["title_clean"] = df["title"].apply(clean_text)

df = df[df["text_clean"].str.len() > 10]
df = df.dropna(subset=["label"])


In [4]:
df["combined"] = df["title_clean"] + " [SEP] " + df["text_clean"]


In [5]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["combined"].tolist(),
    df["label"].tolist(),
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)



In [6]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-large")


/home/slim/FakeNews/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import torch
from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FakeNewsDataset(train_texts, train_labels, tokenizer)
val_dataset   = FakeNewsDataset(val_texts,   val_labels, tokenizer)
test_dataset  = FakeNewsDataset(test_texts,  test_labels, tokenizer)


In [8]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-large",
    num_labels=2
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

training_args = TrainingArguments(
    output_dir="./roberta_fakenews",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    save_total_limit=2,
    load_best_model_at_end=True,

    learning_rate=2e-5,
    per_device_train_batch_size=1,      # RTX2050 safe
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,     # effective batch = 16
    num_train_epochs=3,
    fp16=True,
    max_grad_norm=1.0
)



In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()

# ===== SAVE BEST MODEL =====
best_model_path = trainer.state.best_model_checkpoint

if best_model_path is None:
    print("⚠️ No best checkpoint found — something went wrong.")
else:
    print(f"📌 Best model checkpoint: {best_model_path}")
    model.save_pretrained("./best_roberta_model")
    tokenizer.save_pretrained("./best_roberta_model")
    print("✅ Best model saved to ./best_roberta_model")


# Evaluate on test dataset
test_metrics = trainer.evaluate(test_dataset)
print(test_metrics)

# Predictions
preds = trainer.predict(test_dataset)
y_true = preds.label_ids
y_pred = preds.predictions.argmax(axis=1)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.126400,0.059795,0.981178,0.981178,0.981178,0.981178
2,0.054300,0.079865,0.983613,0.983675,0.983613,0.983614


TrainOutput(global_step=6230, training_loss=0.09035712620228481, metrics={'train_runtime': 4583.6864, 'train_samples_per_second': 43.488, 'train_steps_per_second': 2.718, 'total_flos': 4.644186756008755e+16, 'train_loss': 0.09035712620228481, 'epoch': 2.0})

In [11]:
test_metrics = trainer.evaluate(test_dataset)
print(test_metrics)


{'eval_loss': 0.055455055087804794, 'eval_accuracy': 0.9819271467365859, 'eval_precision': 0.9819317638459829, 'eval_recall': 0.9819271467365859, 'eval_f1': 0.981927517651327, 'eval_runtime': 110.0216, 'eval_samples_per_second': 97.063, 'eval_steps_per_second': 24.268, 'epoch': 2.0}


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

preds = trainer.predict(test_dataset)
y_true = preds.label_ids
y_pred = preds.predictions.argmax(axis=1)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))


              precision    recall  f1-score   support

           0       0.98      0.98      0.98      5254
           1       0.98      0.98      0.98      5425

    accuracy                           0.98     10679
   macro avg       0.98      0.98      0.98     10679
weighted avg       0.98      0.98      0.98     10679

[[5165   89]
 [ 104 5321]]


In [ ]:
OUTPUT_DIR = "../backend/best_roberta_model"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("==> Model saved to", OUTPUT_DIR)
